# Lesson 3.4 — From BC Failure to Modern Robot Learning

这一课把 3.1–3.3 的结论接上"现代方法"：BC 的失败不是实现问题，而是结构问题；DAgger、ACT、Diffusion Policy、VLA 各自修的是这条链上的**不同环节**。

对应 `docs/roadmap_v3.md`：DAgger（3.5）、action chunking / ACT（3.7）、多模态策略与 VLA（3.8 起）。

学完本课后应该能：

1. 说明 Behavior Cloning 为什么在 long-horizon 任务上不够用；
2. 说明 DAgger 如何缓解 distribution shift，以及它的代价；
3. 说明 ACT 为什么预测动作序列，temporal ensembling 在做什么；
4. 说明 Diffusion Policy 为什么学习动作分布；
5. 说明 VLA 把 vision、language、action 合起来解决了什么、**没有**解决什么。

## 1. 回顾：BC 的根本问题

Behavior Cloning 的训练数据来自 expert：

$$
D_E=\{(o_t,a_t)\}_{t=1}^{N},\qquad o_t\sim d_{\pi_E},\qquad a_t\sim\pi_E(\cdot\mid o_t)
$$

学到的策略在**这些 state 上**逼近 expert：

$$
\pi_\theta(o_t)\ \approx\ a_t
$$

它的隐含假设是

$$
d_{\pi_E}(o)=d_{\pi_\theta}(o)
$$

而实际执行时

$$
d_{\pi_E}(o)\neq d_{\pi_\theta}(o)
$$

于是（机制见 3.2）：

```text
small error → different state → unseen observation → larger error → failure
```

误差量级可以写出来。设每一步在自身访问分布上的误差为 $\epsilon$、horizon 为 $T$，经典分析给出

$$
J(\pi_\theta)\ \le\ J(\pi_E)+O(\epsilon T^2)
$$

而 DAgger 把它降到 $O(\epsilon T)$（Ross & Bagnell 2010；Ross, Gordon & Bagnell, AISTATS 2011）。**注意 $T^2$ 与 $T$ 的差别**：horizon 越长，BC 的劣势越大——这解释了为什么长任务上 BC 总是先崩。

## 2. DAgger — 从错误中学习

### 2.1 为什么需要 DAgger？

BC 的数据集是**固定的**：它只包含 expert 访问过的 state。可是部署时 policy 会走到别的地方去，而那些地方没有任何监督信号。

DAgger 的名字来自 **D**ataset **Agg**regation（数据集聚合）。它要回答的问题可以写成一句话：

> 训练分布应该随着 policy 的变化而变化，而不是停在最初的 expert 分布上。

### 2.2 核心想法

一句话：

> **让 policy 自己犯错，然后让 expert 告诉它这些错误状态应该怎么做。**

普通 BC 的流程是单向的：

```text
expert data (o,a), (o,a), (o,a) → train → policy
```

问题在于：policy **从来没有学过"错了之后怎么办"**，因为这些 state 不在数据里。

DAgger 把它变成循环：

```text
Step 1  收集 expert 数据                  D_0
Step 2  训练 BC                          π_1 = train(D_0)
Step 3  让 policy 自己执行（rollout）      o ~ d_{π_1}
Step 4  收集 policy 遇到的新 state         S_1
Step 5  expert 给这些 state 重新标动作      {(o, π_E(o)) : o ∈ S_1}
Step 6  合并进数据集，继续训练              D_1 = D_0 ∪ 新数据
        ↺ 回到 Step 2
```

关键在第 5 步：**标注者是 expert，但 state 是 policy 自己选的**。这正是它缓解 distribution shift 的方式。

### 2.3 数学表示

原始 BC 数据集：

$$
D_0=\{(o,a)\}\quad\text{来自 } d_{\pi_E}
$$

第 $i$ 轮：

$$
\pi_i=\text{train}(D_{i-1})
$$

让 $\pi_i$ 自己 rollout，得到它访问的 state（也就是协变量偏移下的真实输入分布）：

$$
S_i=\{o:\ o\sim d_{\pi_i}\}
$$

交给 expert 打标签：

$$
\tilde D_i=\{(o,\ \pi_E(o)):\ o\in S_i\}
$$

聚合：

$$
D_i=D_{i-1}\cup \tilde D_i
$$

不断迭代，所以叫 **Dataset Aggregation**：数据只增不减，分布则逐步向 $d_{\pi_\theta}$ 靠拢。

一个容易忽略的细节：新数据里的 action 是 expert 的 $a^\star$，**不是** policy 自己的 $\hat a$。DAgger 从不模仿自己的错误动作，它只借用自己到达的 state。

### 2.4 为什么今天不是所有机器人都用 DAgger？

因为它有一个巨大的成本：**需要 expert 在线标注**。

真实机器人上这个循环是：

```text
policy rollout → 发现错误状态 → 人类接管 → 重新示范
```

每一步都要人参与，因此在时间、安全与一致性上都很昂贵；而且数据分布每轮都在变（non-stationary），训练也不稳定。

> **与本项目既有能力的连接。**"人类在关键状态介入并提供正确动作"正是遥操作、VR 演示与 human-in-the-loop 干预能提供的东西。它和本仓库后续要做的 VR demonstration / intervention 数据线是同一件事，而不是额外的工程负担。

因此现代方法寻找成本更低的替代路径：

- **ACT**：减少"一步一步犯错"——改为预测动作序列（§3）；
- **Diffusion Policy**：学习多种可能成功的动作，提高鲁棒性（§4）；
- **VLA**：利用 vision + language + 大规模数据，减少对单个 expert 的依赖（§5）。

### 2.5 两种方法的误差量级

$$
\text{BC}:\quad O(\epsilon T^2) \qquad\qquad \text{DAgger}:\quad O(\epsilon T)
$$

直觉：BC 只在 $d_{\pi_E}$ 上保证 $\epsilon$，一旦偏离，后续每一步都可能遇到"没学过"的 state；DAgger 让训练分布持续追上执行分布，因此误差只随 horizon 线性累加。

代价是它需要 **online expert**——这正是后面几种方法想绕开的部分。

## 3. ACT — Action Chunking with Transformers

BC 的**时间问题**：每个时刻都重新决策一次。

$$
\text{BC}:\quad o_t\rightarrow a_t
$$

例如一次抓取：

```text
move → move → move → grasp → lift
```

每一步独立预测，动作之间没有显式建模；任何一步的小噪声都会立刻成为下一步的输入。

### 3.1 ACT 的做法：预测一个动作块

$$
\text{ACT}:\quad o_t\rightarrow (a_t,a_{t+1},\dots,a_{t+k})
$$

例如一次预测 20 步（$k=20$）的动作序列。结构上是"编码器 + Transformer + 动作头"：

```text
Image + Robot state
        │
        ▼
   Transformer
        │
        ▼
   Action chunk  [a_1, a_2, …, a_20]
```

### 3.2 ACT 解决什么？

1. **Temporal consistency（时间一致性）**：reach → approach → grasp → lift 天然是一段 motion；一次输出整段，比逐步独立预测更稳定。
2. **Reducing decision frequency（降低决策频率）**：100 步从 100 次决策降到 $100/k$ 次（$k=20$ 时 5 次），**减少了预测噪声进入闭环的次数**——这直接对应 3.2 里"误差注入频率"那一项。

一个容易忽略的工程细节：推理时 ACT 通常**每一步都查询一次策略**，再对覆盖当前时刻的多个 chunk 做加权平均（temporal ensembling）：

$$
a_t=\frac{\sum_i w_i\,\hat a_{t,i}}{\sum_i w_i},\qquad w_i=e^{-m\,i}
$$

其中 $\hat a_{t,i}$ 是第 $i$ 个覆盖时刻 $t$ 的预测（越新越靠近 $i=0$），$m$ 控制"多信任新 chunk"。它把 chunk 的平滑性与逐步修正结合起来。

### 3.3 ACT 的局限

ACT 仍然是

$$
o\ \rightarrow\ \text{action}
$$

它**不理解任务**：如果指令是

> "把红色杯子放到桌子上"

ACT 不知道"红色"指哪个物体，也不知道"放到桌子上"意味着什么目标状态——它只是学会了在见过的 state 上输出见过的动作块。

所以要解决的是另外两个问题：

- **action ambiguity（动作不唯一）** → 由 Diffusion Policy（§4）用分布建模处理；
- **task understanding（任务语义）** → 由 VLA（§5）引入语言与视觉。

## 4. Diffusion Policy

### 4.1 为什么需要 Diffusion？

机器人动作**不是唯一的**。例如拿杯子：

```text
方案 A：从左侧抓
方案 B：从右侧抓
```

两者都能成功。但 BC 用 MSE 回归时倾向于输出它们的**平均**：

$$
\pi_\theta(o)\ \approx\ \mathbb E[a\mid o]
$$

而"平均动作"可能谁都抓不到——这是多峰（multimodal）动作分布下的典型失败。

### 4.2 Diffusion Policy 的核心

不是预测一个 $a$，而是学习动作的分布：

$$
p_\theta(a\mid o)\qquad\text{或}\qquad p_\theta(A_t\mid o_t)
$$

即 given observation, **what actions are possible?** 训练沿用去噪扩散（DDPM 形式）：给真实动作块 $A_0$ 加噪得到 $A_t$，让网络预测所加的噪声

$$
\mathcal L(\theta)=\mathbb E_{t,\,A_0,\,\epsilon}\Big[\big\lVert \epsilon-\epsilon_\theta(A_t,\,o_t,\,t)\big\rVert^2\Big],
\qquad A_t=\sqrt{\bar\alpha_t}\,A_0+\sqrt{1-\bar\alpha_t}\,\epsilon
$$

采样时从噪声出发逐步去噪：

```text
noise action → denoise × N → trajectory (action chunk)
```

这与图像生成是同一套机制，只是把"像素"换成了"动作序列"。

### 4.3 优势与代价

适合：

- **multimodal action**：多个可行解不必被平均掉；
- **complex manipulation**、**contact-rich tasks**：这些任务的"正确动作"往往是多峰的。

代价是采样需要多步去噪、推理更慢。工程上用 **receding horizon** 缓解——一次预测 $H$ 步，只执行前 $k$ 步再重新观测：

$$
\text{execute }(a_t,\dots,a_{t+k-1})\ \text{of}\ (a_t,\dots,a_{t+H-1}),\qquad k<H
$$

## 5. VLA — Vision Language Action

前面三种方法都把输入限制在 **state**（或 state + 图像）：

| 方法 | 映射 |
|---|---|
| BC | state → action |
| ACT | state → action sequence |
| Diffusion | state → action distribution |

它们共同缺少的是 **task understanding**：任务是什么、涉及哪个物体、目标状态是什么、之前发生过什么。语言指令恰好承载这部分信息。

### 为什么需要 VLA？

语言提供了三件前面方法没有的东西：

1. **任务身份**：同一个场景下，"pick the red cube" 与 "push the blue box" 是不同的任务；
2. **可组合的目标**：语言可以描述训练集中没有出现过的组合；
3. **可迁移的语义**：vision-language 预训练把大量外部知识带进 policy，减少对单一 expert 数据集的依赖。

### VLA 的结构

```text
            Image ──┐
                    │
Language instruction ──► VLM backbone ──► Action head ──► Robot
                    │
        Robot state ──┘
```

关键设计选择在**动作头**：

- **离散 action token**（RT-2、OpenVLA）：把动作当作另一种"词"，复用语言模型的自回归接口，能继承语义知识，代价是要经过量化；
- **连续生成头**（diffusion / flow matching，例如 `π0` 的 action expert）：输出连续动作块，保留精度与多峰性，代价是结构更重。

无论哪种，都绕不开本仓库反复强调的前提：**action 的语义必须明确**——coordinate frame、绝对/增量、单位、控制模式、维度。否则跨机器人数据无法真正混合（参见 `1.2_action_space_and_control_modes.ipynb` 与 `notes/concepts.md`）。

### VLA 没有解决什么

它改善的是**泛化到新任务/新物体**，而不是误差累积本身：VLA 仍然需要一个动作头在闭环里执行，仍然会经历 3.2 的协变量偏移。因此 VLA 的成功率同样必须用 3.3 的 closed-loop 协议来衡量，而不是只看 action prediction error。

## 6. 方法演化总结

| 方法 | 解决的问题 | 通过什么机制 | **没有**解决什么 |
|---|---|---|---|
| **BC** | 学习 expert action | 在 $D_E$ 上做监督学习 | 分布偏移、多峰动作、任务理解 |
| **DAgger** | distribution shift | 用 policy 访问的 state 让 expert 在线标注 | 标注成本、多峰、任务理解 |
| **ACT** | temporal consistency、决策频率 | 预测 action chunk（+ temporal ensembling） | 分布偏移、任务理解 |
| **Diffusion Policy** | action ambiguity | 学习 $p_\theta(a\mid o)$ | 任务理解、推理成本 |
| **VLA** | semantic understanding | vision + language + 大规模预训练 | 长时程 task state 跟踪与恢复（见 §小结） |

这张表值得反复看：**每个方法只修链条上的一环**，没有哪一个单独构成完整的 embodied agent。

## 小结

Behavior Cloning 失败的根本原因是训练 state 与部署 state 不同：$d_{\pi_E}\neq d_{\pi_\theta}$。现代方法各自针对不同的限制：

- **BC**：学习 expert action；
- **DAgger**：收集 policy 诱导的 state，把误差量级从 $O(\epsilon T^2)$ 降到 $O(\epsilon T)$；
- **ACT**：预测时间一致的 action chunk，降低决策频率；
- **Diffusion Policy**：学习多峰动作分布 $p_\theta(a\mid o)$；
- **VLA**：把 vision、language、action 合起来，获得任务语义与泛化。

但要注意这张图里**缺的那一块**：它们都在改进"如何把 observation 映射成 action"，没有哪一个负责"任务进行到哪一步、之前发生了什么、失败后如何恢复"。这正是本项目的长期方向（task state + memory + task world model），也是 Lesson 3.5 之后逐层展开的内容。

一个完整的 embodied agent 需要：

$$
\text{task reasoning}\ +\ \text{memory}\ +\ \text{policy execution}
$$

而其中每一项都必须用 closed-loop 成功率（3.3）而不是 offline loss 来衡量。